# 03 — Bias Audit

This is the core analysis of the project. We take the baseline model's predictions  
and examine whether pricing errors are distributed unevenly across demographic groups.

**Audit dimensions:**
1. Mean prediction error by group (systematic over/under-prediction)
2. RMSE by group (accuracy disparity)
3. Effective price-per-mile by group (economic impact)
4. Error distribution visualizations

**Sensitive attributes examined:**
- `majority_race` — majority-white vs. majority-Black vs. mixed tracts
- `income_group` — income quartiles of the pickup neighborhood

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error
import os
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = '../data'
FIG_DIR = '../figures'
os.makedirs(FIG_DIR, exist_ok=True)

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

In [ ]:
# ── Load test predictions ────────────────────────────────────────────────────
df = pd.read_csv(os.path.join(DATA_DIR, 'test_with_predictions.csv'))

df['error'] = df['predicted_fare'] - df['fare']             # positive = overcharged
df['abs_error'] = np.abs(df['error'])
df['pct_error'] = df['error'] / df['fare'] * 100
df['predicted_ppm'] = df['predicted_fare'] / df['trip_miles']  # predicted price per mile

print(f'Loaded {len(df):,} test predictions')
print(f"Groups: {df['majority_race'].value_counts().to_dict()}")

## 3.1 — Mean Prediction Error by Race Group

A positive mean error means the model systematically **overestimates** fares  
for trips originating in that type of neighborhood.

In [ ]:
race_metrics = df.groupby('majority_race').agg(
    n_trips=('fare', 'count'),
    mean_actual_fare=('fare', 'mean'),
    mean_predicted_fare=('predicted_fare', 'mean'),
    mean_error=('error', 'mean'),
    mean_abs_error=('abs_error', 'mean'),
    rmse=('error', lambda x: np.sqrt(np.mean(x**2))),
    mean_pct_error=('pct_error', 'mean'),
    mean_actual_ppm=('price_per_mile', 'mean'),
    mean_predicted_ppm=('predicted_ppm', 'mean'),
).round(3)

print('=== Bias Audit: Metrics by Majority Race of Pickup Tract ===')
race_metrics

In [ ]:
# ── Bar chart: mean error by group ───────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Mean error
race_metrics['mean_error'].plot(kind='bar', ax=axes[0], color=['#4c72b0', '#dd8452', '#55a868'])
axes[0].set_title('Mean Prediction Error ($)')
axes[0].set_ylabel('Error (pred − actual)')
axes[0].axhline(0, color='black', lw=0.8, ls='--')
axes[0].tick_params(axis='x', rotation=15)

# RMSE
race_metrics['rmse'].plot(kind='bar', ax=axes[1], color=['#4c72b0', '#dd8452', '#55a868'])
axes[1].set_title('RMSE by Group ($)')
axes[1].set_ylabel('RMSE')
axes[1].tick_params(axis='x', rotation=15)

# Mean predicted price per mile
race_metrics[['mean_actual_ppm', 'mean_predicted_ppm']].plot(kind='bar', ax=axes[2])
axes[2].set_title('Price per Mile: Actual vs Predicted')
axes[2].set_ylabel('$/mile')
axes[2].tick_params(axis='x', rotation=15)
axes[2].legend(['Actual', 'Predicted'])

fig.suptitle('Baseline Model — Bias Audit by Neighborhood Demographics', fontsize=14, y=1.02)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'audit_race_bars.png'), dpi=150, bbox_inches='tight')
plt.show()

## 3.2 — Error Distribution by Group

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot of raw errors
sns.boxplot(data=df, x='majority_race', y='error', ax=axes[0],
            showfliers=False)  # hide outliers for readability
axes[0].axhline(0, color='red', lw=1, ls='--')
axes[0].set_title('Distribution of Prediction Errors')
axes[0].set_ylabel('Error: predicted − actual ($)')
axes[0].set_xlabel('')

# Violin plot of percentage errors
plot_df = df[df['pct_error'].between(-100, 100)]  # clip for readability
sns.violinplot(data=plot_df, x='majority_race', y='pct_error', ax=axes[1], cut=0)
axes[1].axhline(0, color='red', lw=1, ls='--')
axes[1].set_title('Distribution of Percentage Errors')
axes[1].set_ylabel('% Error')
axes[1].set_xlabel('')

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'audit_error_distributions.png'), dpi=150)
plt.show()

## 3.3 — Audit by Income Group

Income quartiles provide a second lens — geographic income level may correlate  
with service patterns differently than race.

In [ ]:
income_order = ['low_income', 'lower_mid', 'upper_mid', 'high_income']
df_inc = df[df['income_group'].isin(income_order)].copy()

income_metrics = df_inc.groupby('income_group').agg(
    n_trips=('fare', 'count'),
    mean_error=('error', 'mean'),
    rmse=('error', lambda x: np.sqrt(np.mean(x**2))),
    mean_pct_error=('pct_error', 'mean'),
).reindex(income_order).round(3)

print('=== Bias Audit: Metrics by Income Group ===')
income_metrics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

income_metrics['mean_error'].plot(kind='bar', ax=axes[0], color='#7a76c2')
axes[0].set_title('Mean Prediction Error by Income Quartile')
axes[0].set_ylabel('Error ($)')
axes[0].axhline(0, color='black', lw=0.8, ls='--')
axes[0].tick_params(axis='x', rotation=15)

income_metrics['rmse'].plot(kind='bar', ax=axes[1], color='#c2767a')
axes[1].set_title('RMSE by Income Quartile')
axes[1].set_ylabel('RMSE ($)')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'audit_income_bars.png'), dpi=150)
plt.show()

## 3.4 — Disparate Impact Ratio

We adapt the classic disparate impact measure (from classification) to regression:  
compare the mean predicted fare between the most- and least-favored groups.

A ratio far from 1.0 suggests the model treats groups differently in aggregate.

In [ ]:
# Disparate impact on mean absolute error
group_mae = df.groupby('majority_race')['abs_error'].mean()

if len(group_mae) >= 2:
    best_group = group_mae.idxmin()
    worst_group = group_mae.idxmax()
    di_ratio = group_mae[best_group] / group_mae[worst_group]
    print(f'Best-served group:  {best_group}  (MAE = {group_mae[best_group]:.3f})')
    print(f'Worst-served group: {worst_group}  (MAE = {group_mae[worst_group]:.3f})')
    print(f'Disparate Impact Ratio (MAE): {di_ratio:.3f}')
    print(f'  (1.0 = perfect parity, <0.8 is commonly considered problematic)')

## 3.5 — Summary Table for Report

A single consolidated table suitable for inclusion in the final report.

In [ ]:
summary = race_metrics[['n_trips', 'mean_error', 'rmse', 'mean_pct_error', 'mean_predicted_ppm']].copy()
summary.columns = ['N Trips', 'Mean Error ($)', 'RMSE ($)', 'Mean % Error', 'Pred. $/mile']

print('\n=== Summary Table: Baseline Model Bias Audit ===')
print(summary.to_string())

# Save for report
summary.to_csv(os.path.join(DATA_DIR, 'audit_summary_baseline.csv'))
print(f'\nSaved to {DATA_DIR}/audit_summary_baseline.csv')